In [ ]:
"""
═══════════════════════════════════════════════════════════════
 SAM FINANCIAL MODEL - NOTEBOOK DE ANÁLISE E INSIGHTS
═══════════════════════════════════════════════════════════════
 
 Objetivo: Gerar insights profundos, analisar cenários e testar mitigações
 para a startup SAM, utilizando os módulos config.py e engine.py.
 
 Autor: Seu Arquiteto de Finanças & Cientista de Dados
═══════════════════════════════════════════════════════════════
"""

# ═══════════════════════════════════════════════════════════════
# 1. IMPORTAÇÕES E CONFIGURAÇÕES
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Módulos do projeto
from config import (
    ConfigFinanceira, 
    CONFIG_PADRAO, 
    CONFIG_PESSIMISTA, 
    CONFIG_OTIMISTA,
    criar_config_com_contratacoes
)
from engine import MotorProjecaoFinanceira

# Configurações de visualização e exibição
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)

print("✅ Ambiente configurado com sucesso!")

# ═══════════════════════════════════════════════════════════════
# 2. DEFINIÇÃO E EXECUÇÃO DO CENÁRIO BASE (REALISTA)
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 CENÁRIO BASE: REALISTA (COM CONTRATAÇÕES)")
print("="*80)

# Usaremos a configuração com contratações planejadas como nosso "realista"
config_base = criar_config_com_contratacoes()

print("\n📋 Premissas do Cenário Base:")
print(f"  • Aporte Mensal: R$ {config_base.aporte_mensal_fixo:,.2f}")
print(f"  • ARPU: R$ {config_base.arpu_medio:,.2f}")
print(f"  • Crescimento de Tráfego: {config_base.taxa_crescimento_trafego_mensal*100:.0f}%/mês")
print(f"  • Churn: {config_base.churn_mensal*100:.1f}%/mês")
print(f"  • Funcionários: {len(config_base.equipe)} pessoa(s) contratada(s)")

# Executa a projeção
motor_base = MotorProjecaoFinanceira(config_base)
projecao_base_df = motor_base.executar_projecao(meses=36)
kpis_base = motor_base.calcular_kpis()

print("\n🚀 Projeção executada. Exibindo relatório de viabilidade:")
print(motor_base.gerar_relatorio_texto())

# ═══════════════════════════════════════════════════════════════
# 3. ANÁLISE VISUAL DO CENÁRIO BASE
# ═══════════════════════════════════════════════════════════════

def plotar_dashboard_cenario(df_projecao, titulo="Dashboard Financeiro"):
    """Gera um dashboard com os gráficos mais importantes."""
    fig, axs = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle(titulo, fontsize=20, fontweight='bold')

    # 1. Crescimento de Usuários e MRR
    ax1 = axs[0, 0]
    ax1_twin = ax1.twinx()
    ax1.plot(df_projecao['Mes'], df_projecao['Usuarios_Finais'], color='g', label='Usuários Finais')
    ax1_twin.plot(df_projecao['Mes'], df_projecao['MRR'], color='b', label='MRR')
    ax1.set_xlabel('Mês')
    ax1.set_ylabel('Usuários Finais', color='g')
    ax1_twin.set_ylabel('MRR (R$)', color='b')
    ax1.set_title('Crescimento de Usuários e MRR')
    ax1.grid(True, alpha=0.3)

    # 2. Fluxo de Caixa e Vale da Morte
    ax2 = axs[0, 1]
    ax2.fill_between(df_projecao['Mes'], df_projecao['Saldo_Caixa'], color='skyblue', alpha=0.4)
    ax2.plot(df_projecao['Mes'], df_projecao['Saldo_Caixa'], color='Slateblue', alpha=0.8, lw=2)
    ax2.axhline(0, color='red', linestyle='--', lw=2)
    ax2.set_xlabel('Mês')
    ax2.set_ylabel('Saldo de Caixa (R$)')
    ax2.set_title('Evolução do Fluxo de Caixa')
    ax2.grid(True, alpha=0.3)

    # 3. Composição de Custos (OPEX)
    ax3 = axs[0, 2]
    # Seleciona apenas as colunas de custo para o gráfico de área empilhada
    custos_cols = ['Custo_Infra', 'Custo_Marketing', 'Salario_Pago']
    # Adiciona custos de pessoal, ferramentas, etc. se existirem no DF
    # Nota: O engine atual agrupa tudo em OPEX_Total. Para detalhar, 
    # o engine precisaria retornar um DF com cada custo separado.
    # Por agora, vamos mostrar os 3 principais que já estão no DF.
    df_projecao.plot(x='Mes', y=custos_cols, kind='area', stacked=True, ax=ax3, colormap='viridis')
    ax3.set_xlabel('Mês')
    ax3.set_ylabel('Custo (R$)')
    ax3.set_title('Composição dos Custos Principais (OPEX)')
    ax3.grid(True, alpha=0.3)

    # 4. Receita vs. Lucro Bruto
    ax4 = axs[1, 0]
    ax4.plot(df_projecao['Mes'], df_projecao['MRR'], label='MRR (Receita)', lw=3)
    ax4.plot(df_projecao['Mes'], df_projecao['Lucro_Bruto'], label='Lucro Bruto', lw=3)
    ax4.set_xlabel('Mês')
    ax4.set_ylabel('Valor (R$)')
    ax4.set_title('Receita vs. Lucro Bruto')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # 5. Evolução do CAC e LTV/CAC
    ax5 = axs[1, 1]
    ax5_twin = ax5.twinx()
    ax5.plot(df_projecao['Mes'], df_projecao['CAC_Mensal'], 'o-', color='r', label='CAC Mensal')
    ax5_twin.plot(df_projecao['Mes'], df_projecao['LTV_CAC_Ratio_Mensal'], 'o-', color='purple', label='LTV/CAC Ratio')
    ax5_twin.axhline(3, color='gray', linestyle='--', label='Meta LTV/CAC (3x)')
    ax5.set_xlabel('Mês')
    ax5.set_ylabel('CAC (R$)', color='r')
    ax5_twin.set_ylabel('LTV/CAC Ratio', color='purple')
    ax5.set_title('Eficiência de Aquisição (CAC e LTV/CAC)')
    ax5.grid(True, alpha=0.3)

    # 6. Resumo de KPIs por Ano
    ax6 = axs[1, 2]
    ax6.axis('tight')
    ax6.axis('off')
    kpi_data = [
        ['KPI', 'Ano 1', 'Ano 2', 'Ano 3'],
        ['MRR (R$)', f"{df_projecao.iloc[11]['MRR']:,.0f}", f"{df_projecao.iloc[23]['MRR']:,.0f}", f"{df_projecao.iloc[35]['MRR']:,.0f}"],
        ['Usuários', f"{df_projecao.iloc[11]['Usuarios_Finais']:.0f}", f"{df_projecao.iloc[23]['Usuarios_Finais']:.0f}", f"{df_projecao.iloc[35]['Usuarios_Finais']:.0f}"],
        ['Saldo Caixa (R$)', f"{df_projecao.iloc[11]['Saldo_Caixa']:,.0f}", f"{df_projecao.iloc[23]['Saldo_Caixa']:,.0f}", f"{df_projecao.iloc[35]['Saldo_Caixa']:,.0f}"]
    ]
    table = ax6.table(cellText=kpi_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)
    ax6.set_title('Resumo de Desempenho Anual', fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

print("\n📈 Gerando dashboard visual do cenário base...")
plotar_dashboard_cenario(projecao_base_df, "Dashboard - Cenário Base (Realista)")


# ═══════════════════════════════════════════════════════════════
# 4. ANÁLISE COMPARATIVA DE CENÁRIOS (VERSÃO CORRIGIDA)
# ═══════════════════════════════════════════════════════════════

def analisar_e_comparar_cenarios():
    """Executa e compara os 3 cenários principais (Pessimista, Realista, Otimista)."""
    
    cenarios = {
        'Pessimista': CONFIG_PESSIMISTA,
        'Realista': criar_config_com_contratacoes(), # Usar o mesmo do cenário base
        'Otimista': CONFIG_OTIMISTA
    }
    
    resultados_completos = {}
    resumo_kpis = []
    
    for nome, config_cenario in cenarios.items():
        motor_temp = MotorProjecaoFinanceira(config_cenario)
        df_temp = motor_temp.executar_projecao(36)
        kpis_temp = motor_temp.calcular_kpis()
        
        resultados_completos[nome] = df_temp
        
        # [CORREÇÃO E MELHORIA] Usando .get() para evitar KeyErrors e fornecer valores padrão
        resumo_kpis.append({
            'Cenário': nome,
            'MRR Ano 3 (R$)': kpis_temp.get('MRR_Ano3', 0),
            'Usuários Ano 3': int(kpis_temp.get('Usuarios_Ano3', 0)),
            'Saldo Caixa Final (R$)': kpis_temp.get('Saldo_Caixa_Final', 0),
            'Vale da Morte (R$)': kpis_temp.get('Vale_da_Morte_Minimo_Caixa', 0),
            # Se o Break-Even não for atingido, ele retorna None. Usamos 36 como padrão para a tabela.
            'Break-Even (Mês)': kpis_temp.get('Break_Even_Mes') or 36,
            # [CORREÇÃO PRINCIPAL] Chave corrigida de 'LTV_CAC_Ratio' para 'LTV_CAC_Ratio_Final'
            'LTV/CAC': kpis_temp.get('LTV_CAC_Ratio_Final', 0)
        })
    
    df_comparacao = pd.DataFrame(resumo_kpis).set_index('Cenário')
    
    print("\n📋 Tabela Comparativa de KPIs:")
    # Formatação para melhor visualização
    display(df_comparacao.style.format({
        "MRR Ano 3 (R$)": "R$ {:,.0f}",
        "Usuários Ano 3": "{:,}",
        "Saldo Caixa Final (R$)": "R$ {:,.0f}",
        "Vale da Morte (R$)": "R$ {:,.0f}",
        "LTV/CAC": "{:.2f}x"
    }))

    # Gráfico comparativo
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Comparação Visual dos Cenários', fontsize=18, fontweight='bold')
    
    df_comparacao['MRR Ano 3 (R$)'].plot(kind='bar', ax=axes[0], color=['#e74c3c', '#f39c12', '#2ecc71'])
    axes[0].set_title('MRR no Ano 3')
    axes[0].set_ylabel('R$')
    
    df_comparacao['Saldo Caixa Final (R$)'].plot(kind='bar', ax=axes[1], color=['#e74c3c', '#f39c12', '#2ecc71'])
    axes[1].set_title('Saldo de Caixa Final (Ano 3)')
    axes[1].set_ylabel('R$')

    df_comparacao['Vale da Morte (R$)'].plot(kind='bar', ax=axes[2], color=['#e74c3c', '#f39c12', '#2ecc71'])
    axes[2].set_title('Vale da Morte (Pior Caixa)')
    axes[2].set_ylabel('R$')
    axes[2].axhline(0, color='black', linestyle='--')
    
    for ax in axes:
        ax.tick_params(axis='x', rotation=0)
        ax.grid(True, alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()
    
    return resultados_completos, df_comparacao

# Agora, execute a função novamente
resultados_cenarios, df_comparacao = analisar_e_comparar_cenarios()


# ═══════════════════════════════════════════════════════════════
# 5. ANÁLISE DE SENSIBILIDADE (QUAL O MAIOR VILÃO?)
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 ANÁLISE DE SENSIBILIDADE: IMPACTO DO CHURN")
print("="*80)
print("Analisando como a taxa de cancelamento (churn) destrói o valor do negócio (LTV).")

def analisar_sensibilidade_churn():
    churns = np.arange(0.02, 0.09, 0.01) # De 2% a 8%
    resultados = []
    
    for churn in churns:
        config_temp = ConfigFinanceira(churn_mensal=churn)
        motor_temp = MotorProjecaoFinanceira(config_temp)
        motor_temp.executar_projecao(36)
        kpis_temp = motor_temp.calcular_kpis()
        
        resultados.append({
            'Churn (%)': f"{churn*100:.0f}%",
            'LTV (R$)': kpis_temp['LTV_Final'],
            'LTV/CAC': kpis_temp['LTV_CAC_Ratio_Final'],
            'MRR Ano 3 (R$)': kpis_temp['MRR_Ano3']
        })
    
    df_sens = pd.DataFrame(resultados)
    
    # Gráfico do impacto
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Sensibilidade do Negócio ao Churn', fontsize=16, fontweight='bold')
    
    sns.lineplot(data=df_sens, x='Churn (%)', y='LTV (R$)', ax=ax1, marker='o', lw=3, color='#e74c3c')
    ax1.set_title('Impacto do Churn no LTV')
    ax1.set_ylabel('LTV (R$)')
    ax1.grid(True, alpha=0.3)

    sns.lineplot(data=df_sens, x='Churn (%)', y='LTV/CAC', ax=ax2, marker='o', lw=3, color='#3498db')
    ax2.axhline(3, color='red', linestyle='--', lw=2, label='Meta Saldável (3x)')
    ax2.set_title('Impacto do Churn na Relação LTV/CAC')
    ax2.set_ylabel('LTV/CAC Ratio')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.show()
    
    print("\n📋 Tabela de Sensibilidade ao Churn:")
    display(df_sens)
    
    return df_sens

df_sensibilidade = analisar_sensibilidade_churn()


# ═══════════════════════════════════════════════════════════════
# 6. ANÁLISE DE MITIGAÇÃO (CENÁRIO "E SE...?")
# ═══════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("🛠️ ANÁLISE DE MITIGAÇÃO: TESTANDO UMA ESTRATÉGIA")
print("="*80)
print("PROBLEMA IDENTIFICADO: O CAC está alto e o Payback é longo no cenário base.")
print("MITIGAÇÃO PROPOSTA: Vamos investir mais em marketing para acelerar o crescimento,")
print("                e testar se isso compensa o maior gasto no curto prazo.\n")

# Criar a configuração da mitigação
config_mitigacao = criar_config_com_contratacoes()
config_mitigacao.marketing_fase2_perc_lucro_bruto = 0.40 # Aumenta o reinvestimento de 25% para 40%
config_mitigacao.taxa_crescimento_trafego_mensal = 0.25 # Aumenta o crescimento de 20% para 25%

print("📋 Premissas da Mitigação:")
print(f"  • Reinvestimento em Marketing: {config_mitigacao.marketing_fase2_perc_lucro_bruto*100:.0f}% do Lucro Bruto")
print(f"  • Crescimento de Tráfego: {config_mitigacao.taxa_crescimento_trafego_mensal*100:.0f}%/mês")

# Executar a projeção da mitigação
motor_mitigacao = MotorProjecaoFinanceira(config_mitigacao)
projecao_mitigacao_df = motor_mitigacao.executar_projecao(36)
kpis_mitigacao = motor_mitigacao.calcular_kpis()

# Comparar Base vs. Mitigação
df_mitigacao = pd.DataFrame([
    {
        'Métrica': 'MRR Ano 3 (R$)',
        'Cenário Base': kpis_base['MRR_Ano3'],
        'Com Mitigação': kpis_mitigacao['MRR_Ano3'],
        'Variação (%)': (kpis_mitigacao['MRR_Ano3'] / kpis_base['MRR_Ano3'] - 1) * 100
    },
    {
        'Métrica': 'Saldo Caixa Final (R$)',
        'Cenário Base': kpis_base['Saldo_Caixa_Final'],
        'Com Mitigação': kpis_mitigacao['Saldo_Caixa_Final'],
        'Variação (%)': (kpis_mitigacao['Saldo_Caixa_Final'] / kpis_base['Saldo_Caixa_Final'] - 1) * 100
    },
    {
        'Métrica': 'Vale da Morte (R$)',
        'Cenário Base': kpis_base['Vale_da_Morte_Minimo_Caixa'],
        'Com Mitigação': kpis_mitigacao['Vale_da_Morte_Minimo_Caixa'],
        'Variação (%)': (kpis_mitigacao['Vale_da_Morte_Minimo_Caixa'] / kpis_base['Vale_da_Morte_Minimo_Caixa'] - 1) * 100
    },
    {
        'Métrica': 'Break-Even (Mês)',
        'Cenário Base': kpis_base['Break_Even_Mes'],
        'Com Mitigação': kpis_mitigacao['Break_Even_Mes'],
        'Variação (%)': f"{kpis_mitigacao['Break_Even_Mes'] - kpis_base['Break_Even_Mes']} meses"
    }
])

print("\n📋 Comparativo: Base vs. Mitigação")
display(df_mitigacao)

# Gráfico comparativo de fluxo de caixa
plt.figure(figsize=(14, 7))
plt.plot(projecao_base_df['Mes'], projecao_base_df['Saldo_Caixa'], label='Cenário Base', lw=3)
plt.plot(projecao_mitigacao_df['Mes'], projecao_mitigacao_df['Saldo_Caixa'], label='Com Mitigação', lw=3, linestyle='--')
plt.axhline(0, color='red', linestyle=':', lw=2)
plt.title('Comparativo de Fluxo de Caixa: Base vs. Mitigação', fontweight='bold', fontsize=16)
plt.xlabel('Mês')
plt.ylabel('Saldo de Caixa (R$)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n💡 INSIGHT DA MITIGAÇÃO:")
print("A estratégia de acelerar o crescimento gera um MRR significativamente maior no longo prazo,")
print("mas aprofunda o 'vale da morte' no curto prazo. A decisão de adotá-la depende do seu")
print("apetite ao risco e da pressão dos investidores por crescimento acelerado.")

print("\n" + "="*80)
print("🎉 ANÁLISE CONCLUÍDA COM SUCESSO!")
print("Explore outros cenários e mitigações modificando as configurações acima.")
print("="*80)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ANÁLISE VISUAL MELHORADA E EXPLICATIVA
# ═══════════════════════════════════════════════════════════════

# Garante que os dados necessários existem
if 'projecao_base_df' not in locals() or 'kpis_base' not in locals():
    print("⚠️ Execute a célula das tabelas primeiro para garantir que os dados existam.")

# --- 1. ANÁLISE DE SENSIBILIDADE AO CHURN (COM NARRATIVA) ---
print("\n" + "="*80)
print("📊 ANÁLISE DE SENSIBILIDADE: O IMPACTO DO CHURN NO SEU NEGÓCIO")
print("="*80)
print("O Churn (taxa de cancelamento) é um dos maiores vilões de um SaaS.")
print("Veja como um pequeno aumento pode destruir a rentabilidade do negócio.\n")

def plotar_sensibilidade_explicativa():
    churns = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08]
    resultados = []
    
    for churn in churns:
        config_temp = ConfigFinanceira(churn_mensal=churn)
        motor_temp = MotorProjecaoFinanceira(config_temp)
        motor_temp.executar_projecao(36)
        kpis_temp = motor_temp.calcular_kpis()
        resultados.append(kpis_temp)

    # Cria o gráfico
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle('Análise de Sensibilidade: Como o Churn Destrói o Valor do Cliente', fontsize=18, fontweight='bold')

    # Gráfico 1: Impacto no LTV
    churns_pct = [f"{c*100:.0f}%" for c in churns]
    ltvs = [r['LTV_Final'] for r in resultados]
    
    bars1 = ax1.bar(churns_pct, ltvs, color=['#2ecc71', '#2ecc71', '#f39c12', '#f39c12', '#e67e22', '#e74c3c'])
    ax1.set_ylabel('Lifetime Value (LTV) em Reais', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Taxa de Churn Mensal', fontsize=12, fontweight='bold')
    ax1.set_title('Quanto cada cliente vale para você?', fontsize=14)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Formata o eixo Y para mostrar em milhares (k)
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R$ {x/1000:.0f}k'))
    
    # Adiciona a explicação no ponto crítico
    valor_churn_8 = ltvs[-1]
    ax1.text("8%", valor_churn_8 + 1000, 
             f'Com 8% de churn,\n cada cliente vale\n apenas R$ {valor_churn_8:,.0f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold', color='white',
             bbox=dict(boxstyle='round,pad=0.5', fc='#e74c3c', alpha=0.8))

    # Gráfico 2: Impacto na Relação LTV/CAC
    ltv_cac_ratios = [r['LTV_CAC_Ratio_Final'] for r in resultados]
    
    bars2 = ax2.bar(churns_pct, ltv_cac_ratios, color=['#2ecc71', '#2ecc71', '#f39c12', '#f39c12', '#e67e22', '#e74c3c'])
    ax2.set_ylabel('Relação LTV / CAC (em vezes)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Taxa de Churn Mensal', fontsize=12, fontweight='bold')
    ax2.set_title('A saúde do seu negócio (LTV/CAC)', fontsize=14)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(3, color='blue', linestyle='--', lw=3, label='Meta Saudável (3.0x)')
    ax2.legend()

    # Adiciona a explicação sobre a meta
    ax2.text(0.5, 3.5, 
             'A meta é LTV/CAC > 3.0x.\n Abaixo disso, você gasta demais\n para adquirir um cliente.',
             ha='center', va='center', fontsize=11, color='blue',
             bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plotar_sensibilidade_explicativa()


# --- 2. COMPARAÇÃO DE CENÁRIOS (VISUAL CLARO) ---
print("\n" + "="*80)
print("📊 COMPARAÇÃO DE CENÁRIOS: ONDE SEU NEGÓCIO PODE CHEGAR?")
print("="*80)
print("Compare o futuro pessimista, realista e otimista da sua startup.\n")

def plotar_comparacao_explicativa():
    # Executa os cenários (se ainda não foram executados)
    cenarios_nomes = ['Pessimista', 'Realista', 'Otimista']
    configs_cenarios = [CONFIG_PESSIMISTA, criar_config_com_contratacoes(), CONFIG_OTIMISTA]
    resultados_cenarios = {}
    
    for nome, config in zip(cenarios_nomes, configs_cenarios):
        motor_temp = MotorProjecaoFinanceira(config)
        df_temp = motor_temp.executar_projecao(36)
        kpis_temp = motor_temp.calcular_kpis()
        resultados_cenarios[nome] = {'df': df_temp, 'kpis': kpis_temp}

    # Gráfico Comparativo de MRR no Ano 3
    fig, ax = plt.subplots(figsize=(12, 7))
    mrr_ano3 = [resultados_cenarios[n]['kpis']['MRR_Ano3'] for n in cenarios_nomes]
    cores = ['#e74c3c', '#f39c12', '#2ecc71']
    
    bars = ax.bar(cenarios_nomes, mrr_ano3, color=cores)
    
    # Adiciona os valores no topo das barras
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2.0, yval + 5000, f'R$ {yval/1000:.0f}k', 
                ha='center', va='bottom', fontsize=12, fontweight='bold')

    ax.set_ylabel('Receita Mensal Recorrente (MRR) no Ano 3', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cenário de Negócio', fontsize=12, fontweight='bold')
    ax.set_title('Comparação de Receita (MRR) no 3º Ano', fontsize=16, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Formata o eixo Y
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R$ {x/1000:.0f}k'))
    
    # Remove as bordas desnecessárias
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()

plotar_comparacao_explicativa()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TABELAS DE PROJEÇÃO DETALHADAS (PARA IMPRESSÃO E ANEXO)
# ═══════════════════════════════════════════════════════════════

# Garante que a projeção base foi executada. Se não, execute-a.
if 'projecao_base_df' not in locals():
    print("⚠️ Executando a projeção base primeiro...")
    config_base = criar_config_com_contratacoes()
    motor_base = MotorProjecaoFinanceira(config_base)
    projecao_base_df = motor_base.executar_projecao(meses=36)
    kpis_base = motor_base.calcular_kpis()
    print("✅ Projeção base executada.")

# Função para formatar colunas monetárias
def formatar_monetario(df):
    """Formata colunas de valor monetário para exibição."""
    colunas_monetarias = [
        'MRR', 'COGS', 'Impostos', 'Lucro_Bruto', 
        'Custo_Infra', 'Custo_Marketing', 'Salario_Pago',
        'OPEX_Total', 'Resultado_Operacional', 'Aporte',
        'Fluxo_Caixa', 'Saldo_Caixa', 'CAC_Mensal'
    ]
    df_formatado = df.copy()
    for col in colunas_monetarias:
        if col in df_formatado.columns:
            df_formatado[col] = df_formatado[col].apply(lambda x: f"R$ {x:,.2f}")
    return df_formatado

print("\n" + "="*80)
print("📋 TABELA DE PROJEÇÃO - PRIMEIROS 12 MESES (ANO 1)")
print("="*80)
display(formatar_monetario(projecao_base_df.head(12)))

print("\n" + "="*80)
print("📋 TABELA DE PROJEÇÃO - ÚLTIMOS 12 MESES (ANO 3)")
print("="*80)
display(formatar_monetario(projecao_base_df.tail(12)))

print("\n" + "="*80)
print("📋 TABELA DE PROJEÇÃO COMPLETA (36 MESES)")
print("="*80)
display(formatar_monetario(projecao_base_df))

print("\n💡 DICA: Você pode clicar com o botão direito em uma tabela e selecionar 'Print' para imprimi-la.")